In [ ]:
import os
import git
import numpy as np
from pathlib import Path
from PIL import Image

# =======================
# GLOBAL PROJECT ROOT
# =======================
ROOT_DIR = Path(git.Repo(".", search_parent_directories=True).working_tree_dir)

# =======================
# CONFIG
# =======================
DATASET = "spaceNet"
RAW_DATA_SUFFIX = "spaceNet-cleaned-jitter"
INPUT_DIR = ROOT_DIR / "raw-data" / DATASET / "uncleaned"  # Adjust if raw images are in a different subfolder
OUTPUT_DIR = ROOT_DIR / "raw-data" / DATASET / f"full-{RAW_DATA_SUFFIX}"

CROP_SIZE = 400
SEED = 42
IMG_EXTS = (".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp")

# e.g. SN6_Train_AOI_11_Rotterdam_PS-RGB_20190822134835_20190822135131_tile_4495
def get_bid(path: Path) -> str:
    return path.stem.split("_")[6]


# =======================
# HELPERS
# =======================
def list_images(folder: Path):
    """Return sorted list of image paths in folder (non-recursive)."""
    return sorted(
        folder / f for f in os.listdir(folder)
        if (folder / f).is_file() and (folder / f).suffix.lower() in IMG_EXTS
    )


def crop_non_black_center_rgb(img_rgb: np.ndarray, crop_size: int) -> np.ndarray | None:
    """Crop a crop_size x crop_size window centered on the non-black region (RGB)."""
    non_black = np.any(img_rgb != 0, axis=-1)
    coords = np.argwhere(non_black)
    if coords.size == 0:
        return None

    y0, x0 = coords.min(axis=0)
    y1, x1 = coords.max(axis=0)
    cy = (y0 + y1) // 2
    cx = (x0 + x1) // 2

    h, w, _ = img_rgb.shape
    half = crop_size // 2

    y_start = max(cy - half, 0)
    x_start = max(cx - half, 0)
    y_end = y_start + crop_size
    x_end = x_start + crop_size

    if y_end > h:
        y_end = h
        y_start = max(h - crop_size, 0)
    if x_end > w:
        x_end = w
        x_start = max(w - crop_size, 0)

    cropped = img_rgb[y_start:y_end, x_start:x_end]
    if cropped.shape[0] != crop_size or cropped.shape[1] != crop_size:
        return None
    return cropped


def load_and_crop_rgb(path: Path, crop_size: int) -> np.ndarray | None:
    """Load image as RGB uint8 and crop around non-black region."""
    try:
        img = Image.open(path).convert("RGB")
    except Exception:
        return None
    arr = np.asarray(img)
    return crop_non_black_center_rgb(arr, crop_size)


def compute_batch_stats_rgb(paths: list[Path], crop_size: int):
    """Compute (mean,std) for (R,G,B) over per-image channel means (post-crop)."""
    r_vals, g_vals, b_vals = [], [], []

    for p in paths:
        cropped = load_and_crop_rgb(p, crop_size)
        if cropped is None:
            continue
        cropped_f = cropped.astype(np.float32)
        r_vals.append(cropped_f[..., 0].mean())
        g_vals.append(cropped_f[..., 1].mean())
        b_vals.append(cropped_f[..., 2].mean())

    if not r_vals:
        return (0.0, 0.0, 0.0), (1.0, 1.0, 1.0)

    mean = (float(np.mean(r_vals)), float(np.mean(g_vals)), float(np.mean(b_vals)))
    std = (
        float(np.std(r_vals) or 1.0),
        float(np.std(g_vals) or 1.0),
        float(np.std(b_vals) or 1.0),
    )
    return mean, std


def jitter_and_normalize_rgb(img_rgb_uint8: np.ndarray, mean_rgb, std_rgb, rng):
    """Add uniform jitter then normalize per channel (RGB in/out, float32)."""
    img = img_rgb_uint8.astype(np.float32)
    img += rng.uniform(-0.5, 0.5, size=img.shape).astype(np.float32)

    img[..., 0] = (img[..., 0] - mean_rgb[0]) / std_rgb[0]  # R
    img[..., 1] = (img[..., 1] - mean_rgb[1]) / std_rgb[1]  # G
    img[..., 2] = (img[..., 2] - mean_rgb[2]) / std_rgb[2]  # B
    return img


# =======================
# MAIN
# =======================
def build_full_spacenet_cleaned_jitter(input_dir: Path, output_dir: Path, crop_size: int = 400, seed: int = 42):
    """Crop -> per-batch stats -> jitter+normalize -> save RGB npz into full-{suffix} folder."""
    os.makedirs(output_dir, exist_ok=True)
    paths = list_images(input_dir)

    # group by batch id
    batches = {}
    for p in paths:
        try:
            bid = get_bid(p)
        except Exception:
            continue
        batches.setdefault(bid, []).append(p)

    # precompute stats per batch id
    batch_stats = {bid: compute_batch_stats_rgb(ps, crop_size) for bid, ps in batches.items()}

    rng = np.random.default_rng(seed)
    written = 0

    for p in paths:
        try:
            bid = get_bid(p)
        except Exception:
            continue
        mean_rgb, std_rgb = batch_stats.get(bid, ((0.0, 0.0, 0.0), (1.0, 1.0, 1.0)))

        cropped = load_and_crop_rgb(p, crop_size)
        if cropped is None:
            continue

        out_rgb = jitter_and_normalize_rgb(cropped, mean_rgb, std_rgb, rng)

        # save as .npz with same stem, downstream npz_opener will load first array
        np.savez_compressed(str(output_dir / f"{p.stem}.npz"), image=out_rgb)
        written += 1

    print(f"Wrote {written} npz -> {output_dir}")


if __name__ == "__main__":
    build_full_spacenet_cleaned_jitter(
        input_dir=INPUT_DIR,
        output_dir=OUTPUT_DIR,
        crop_size=CROP_SIZE,
        seed=SEED,
    )


Processed and saved: D:\City Dataset\train\AOI_11_Rotterdam\Extraction\SN6_Train_AOI_11_Rotterdam_PS-RGB_20190804111224_20190804111453_tile_8679.tif
Processed and saved: D:\City Dataset\train\AOI_11_Rotterdam\Extraction\SN6_Train_AOI_11_Rotterdam_PS-RGB_20190804111224_20190804111453_tile_8681.tif
Processed and saved: D:\City Dataset\train\AOI_11_Rotterdam\Extraction\SN6_Train_AOI_11_Rotterdam_PS-RGB_20190804111224_20190804111453_tile_8683.tif
Processed and saved: D:\City Dataset\train\AOI_11_Rotterdam\Extraction\SN6_Train_AOI_11_Rotterdam_PS-RGB_20190804111224_20190804111453_tile_8685.tif
Processed and saved: D:\City Dataset\train\AOI_11_Rotterdam\Extraction\SN6_Train_AOI_11_Rotterdam_PS-RGB_20190804111224_20190804111453_tile_8687.tif
Processed and saved: D:\City Dataset\train\AOI_11_Rotterdam\Extraction\SN6_Train_AOI_11_Rotterdam_PS-RGB_20190804111224_20190804111453_tile_8689.tif
Processed and saved: D:\City Dataset\train\AOI_11_Rotterdam\Extraction\SN6_Train_AOI_11_Rotterdam_PS-RGB_2

KeyboardInterrupt: 